In [3]:
!pip install pypdf gensim nltk

from pypdf import PdfReader
from gensim.models import Word2Vec
from gensim.downloader import load
import nltk
from pathlib import Path
import urllib.request


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 6.4 MB/s eta 0:00:00a 0:00:01


In [17]:
## SET PATH AND DOWNLOAD RESOURCES
nltk.download('punkt')
nltk.download('punkt_tab')

model_path = Path("../models/analogy_solver.model")
model_path.parent.mkdir(parents=True, exist_ok=True)


dataset_path = Path("../../../resources/datasets/questions-words.txt")
dataset_path.parent.mkdir(parents=True, exist_ok=True)

if not dataset_path.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt",
        dataset_path
    )

print(f"Dataset available at: {dataset_path}")



corpus = list(load("text8"))

[nltk_data] Downloading package punkt to /home/jovyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Dataset available at: ../../../resources/datasets/questions-words.txt


In [23]:
## VIEW/ UNDERSTAND DATA

with open(dataset_path,"r") as f:
    lines = f.readlines()

print(f"total # of lines: {len(lines)}")
print(f"top 20 lines look like:\n {lines[:20]}\n\n")

print(f"Sentences: {len(corpus)}")
print(f"First sentence: {corpus[0][:30]}")
print(f"Last sentence: {corpus[-1][:30]}")

total # of lines: 19558
top 20 lines look like:
 [': capital-common-countries\n', 'Athens Greece Baghdad Iraq\n', 'Athens Greece Bangkok Thailand\n', 'Athens Greece Beijing China\n', 'Athens Greece Berlin Germany\n', 'Athens Greece Bern Switzerland\n', 'Athens Greece Cairo Egypt\n', 'Athens Greece Canberra Australia\n', 'Athens Greece Hanoi Vietnam\n', 'Athens Greece Havana Cuba\n', 'Athens Greece Helsinki Finland\n', 'Athens Greece Islamabad Pakistan\n', 'Athens Greece Kabul Afghanistan\n', 'Athens Greece London England\n', 'Athens Greece Madrid Spain\n', 'Athens Greece Moscow Russia\n', 'Athens Greece Oslo Norway\n', 'Athens Greece Ottawa Canada\n', 'Athens Greece Paris France\n', 'Athens Greece Rome Italy\n']


Sentences: 1701
First sentence: ['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english', 'revolution', 'and', 'the', 'sans', 'culottes', 'of', 'the

In [24]:
## TRAIN MODEL & SAVE MODEL

model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=5
)


model.save(str(model_path))
print(f"Saved: {model_path}")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Saved: ../models/analogy_solver.model


In [29]:
## VALIDATION

print(f"""most similar to king: {model.wv.most_similar("king")}\n""")
print(f"""most similar to france: {model.wv.most_similar("france")}\n""")
print(f"""similarity b/w king & queen: {model.wv.similarity("king","queen")}\n""")

most similar to king: [('prince', 0.7372451424598694), ('queen', 0.7196274995803833), ('throne', 0.7068778276443481), ('emperor', 0.7031615376472473), ('sultan', 0.6927268505096436), ('constantine', 0.6821600794792175), ('kings', 0.6816663146018982), ('pope', 0.6811825037002563), ('vii', 0.6802855134010315), ('aragon', 0.6791555881500244)]

most similar to france: [('spain', 0.8213402628898621), ('italy', 0.8164136409759521), ('portugal', 0.7449121475219727), ('belgium', 0.7146454453468323), ('netherlands', 0.6964356899261475), ('catalonia', 0.6871128678321838), ('switzerland', 0.6675021648406982), ('austria', 0.6606177091598511), ('germany', 0.6584458351135254), ('denmark', 0.6550309062004089)]

similarity b/w king & queen: 0.7196274399757385



In [35]:
model.wv.most_similar(
    positive=["paris", "germany"],
    negative=["france"],
    topn=1
)

[('berlin', 0.809036135673523)]

In [38]:
## Most common operations

# | #  | Function                        | Example Call                                                                       | Returns                            |
# | -- | ------------------------------- | ---------------------------------------------------------------------------------- | ---------------------------------- |
# | 1  | Get vector                      | `model.wv["india"]`                                                                | Vector for word                    |
# | 2  | Get vector (explicit)           | `model.wv.get_vector("india")`                                                     | Vector for word                    |
# | 3  | Check existence                 | `"india" in model.wv`                                                              | `True/False`                       |
# | 4  | Most similar words              | `model.wv.most_similar("india", topn=5)`                                           | List of similar words              |
# | 5  | Word analogy                    | `model.wv.most_similar(positive=["king","woman"], negative=["man"], topn=1)`       | `queen`                            |
# | 6  | Similarity                      | `model.wv.similarity("india","pakistan")`                                          | Similarity score                   |
# | 7  | Sentence similarity             | `model.wv.n_similarity(["india","cricket"], ["pakistan","cricket"])`               | Similarity score                   |
# | 8  | Distance                        | `model.wv.distance("india","pakistan")`                                            | Distance score                     |
# | 9  | Odd word out                    | `model.wv.doesnt_match(["india","pakistan","china","banana"])`                     | `"banana"`                         |
# | 10 | Closer words                    | `model.wv.closer_than("india","america")`                                          | Words closer to india than america |
# | 11 | Rank                            | `model.wv.rank("india","pakistan")`                                                | Neighbor rank                      |
# | 12 | Vocabulary words                | `model.wv.index_to_key[:10]`                                                       | Top 10 words                       |
# | 13 | Word index                      | `model.wv.key_to_index["india"]`                                                   | Vocabulary index                   |
# | 14 | Normalized vectors              | `model.wv.get_normed_vectors()`                                                    | All normalized embeddings          |
# | 15 | Vocabulary size                 | `len(model.wv)`                                                                    | Number of words                    |
# | 16 | Save embeddings                 | `model.wv.save_word2vec_format("vectors.txt")`                                     | Writes file                        |
# | 17 | Top-N similar to multiple words | `model.wv.most_similar(positive=["aws","data"], topn=10)`                          | Related words                      |
# | 18 | Positive/Negative analogy       | `model.wv.most_similar(positive=["india","delhi"], negative=["pakistan"], topn=5)` | Analogy result                     |
# | 19 | Raw vocabulary list             | `list(model.wv.key_to_index.keys())`                                               | All words                          |
# | 20 | Vector dimension                | `model.wv.vector_size`                                                             | Embedding size (e.g. 100)          |


[('indonesia', 0.6953096985816956)]

In [9]:
from gensim.models import Word2Vec

model = Word2Vec.load("../models/analogy_solver.model")

print("\nmodel.wv['spain']:")
print(model.wv['spain'])

print("\nmodel.wv.get_vector('spain'):")
print(model.wv.get_vector('spain'))

print("\n'spain' in model.wv:")
print('spain' in model.wv)

print("\nmodel.wv.most_similar('king'):")
print(model.wv.most_similar('king'))

print("\nmodel.wv.most_similar(positive=['king','woman'], negative=['man']):")
print(
    model.wv.most_similar(
        positive=['king', 'woman'],
        negative=['man']
    )
)

print("\nmodel.wv.similarity('king','queen'):")
print(model.wv.similarity('king', 'queen'))

print("\nmodel.wv.distance('king','queen'):")
print(model.wv.distance('king', 'queen'))

print("\nmodel.wv.n_similarity(['king','queen'], ['man','woman']):")
print(
    model.wv.n_similarity(
        ['king', 'queen'],
        ['man', 'woman']
    )
)

print("\nmodel.wv.doesnt_match(['king','queen','prince','banana']):")
print(
    model.wv.doesnt_match(
        ['king', 'queen', 'prince', 'banana']
    )
)

print("\nmodel.wv.closer_than('king','queen')[:10]:")
print(
    model.wv.closer_than(
        'king',
        'queen'
    )[:10]
)

print("\nmodel.wv.rank('king','queen'):")
print(
    model.wv.rank(
        'king',
        'queen'
    )
)

print("\nmodel.wv.index_to_key[:20]:")
print(model.wv.index_to_key[:20])

print("\nmodel.wv.key_to_index['king']:")
print(model.wv.key_to_index['king'])

print("\nlen(model.wv):")
print(len(model.wv))

print("\nmodel.wv.vector_size:")
print(model.wv.vector_size)

print("\nmodel.wv.get_normed_vectors().shape:")
print(model.wv.get_normed_vectors().shape)

print("\nlist(model.wv.key_to_index.keys())[:20]:")
print(list(model.wv.key_to_index.keys())[:20])

print("\nCountry Analogy: paris - france + germany")
print(
    model.wv.most_similar(
        positive=['paris', 'germany'],
        negative=['france']
    )
)

print("\nCountry Analogy: delhi - india + japan")
print(
    model.wv.most_similar(
        positive=['delhi', 'japan'],
        negative=['india']
    )
)

print("\nTechnology Similarity:")
print(
    model.wv.most_similar(
        positive=['computer', 'science']
    )
)


model.wv['spain']:
[-2.200015    0.65117645  0.45506856 -0.6208621   2.05496     0.15465052
 -0.19786729  1.8883783  -0.3368323   2.0530117  -0.09161659 -0.59017515
  0.04405747  0.06804038  2.1037621   1.5420226  -1.2017436   1.690409
  1.6229177  -1.3569186   0.49838454  1.6958507   1.5385969   1.7921203
  1.9726034   0.8163203  -1.7929019  -0.17650925  0.69522756 -2.1680348
 -0.0889749   0.04785709  1.6168603   1.1059437  -1.1590413   1.5485177
  0.29178688 -0.56529605  1.0646315  -0.30933607  1.2079413   1.5996712
 -2.3680182  -0.05445594 -1.0355489   0.3149833  -2.5386968  -0.56260145
 -0.6122034  -0.04888675 -0.37129217  4.134481    0.76512593 -0.20418969
  0.5455848   1.196699   -0.8800805   0.5160397  -0.01802448  0.34897667
 -0.29615498  0.6458925   0.7904914  -1.3501618  -1.1352087   0.10701428
  1.0546595   3.7159305   0.69149166 -1.2361495  -0.5501939  -0.506201
 -1.2815226  -0.64624554 -1.1214343  -2.2379215  -1.7523807   1.262874
 -1.2994868   0.5421697  -0.16954097 -0.2